In [1]:
# 1. Import dan direktori proyek
from pathlib import Path
from datetime import datetime, timezone
from dataclasses import asdict
import json
import sys

import pandas as pd
import torch

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / "multimodal_mil_model.py").is_file() and (p / "app_data_filtered.csv").is_file())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from multimodal_mil_model import ModelConfig, OptimizerConfig
from unimodal_experiments import load_reference_data
from experiment_runner import file_sha256, fingerprint
from task_comparison_experiments import train_comparison, test_comparison, export_paper_comparisons


In [2]:
# 2. Konfigurasi S26; bobot hasil training lama tidak dimuat
VARIANT = "severity"
EXPERIMENT_DIR = ROOT / "experiments" / "Experiment 1"
RUNS_ROOT = EXPERIMENT_DIR / "runs"
REFERENCE_DIR = RUNS_ROOT / "search_20261018_044822_847109_seed1"
REFERENCE_TRIAL = REFERENCE_DIR / "trials" / "S26_pretrained_frozen_adam_P2"
NOTEBOOK_PATH = EXPERIMENT_DIR / "Experiment 1 (MIL-Attention MLP Binary Severity S26).ipynb"
BASELINE_CSV = ROOT / "protocols" / "reference_tables" / "paper_comparison_baselines.csv"
reference = json.loads((REFERENCE_TRIAL / "config.json").read_text(encoding="utf-8"))
MODEL_CONFIG = ModelConfig(**{**reference["model"], "head": "softmax" if VARIANT == "softmax" else "dual_head"})
OPTIMIZER_CONFIG = OptimizerConfig(**reference["optimizer"])
SETTINGS = {**reference["training"], "device": "cuda" if torch.cuda.is_available() else "cpu"}
assert SETTINGS["seed"] == 1 and SETTINGS["monitor"] == "val_f1"
assert SETTINGS["patience"] == 25 and SETTINGS["early_stopping_rule"] == "best_monitor"
assert SETTINGS["epochs"] == 100 and SETTINGS["threshold"] == 0.5
assert MODEL_CONFIG.num_tabular_features == 53 and MODEL_CONFIG.freeze_backbone

# None: training baru. Isi path hanya untuk memakai hasil completed yang identik.
RESUME_RUN_DIR = None
RUN_ID = VARIANT + "_s26_" + datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S_%f") + "_seed1"
RUN_DIR = Path(RESUME_RUN_DIR).resolve() if RESUME_RUN_DIR else RUNS_ROOT / RUN_ID
source_files = ["task_comparison_experiments.py", "unimodal_experiments.py", "binary_training.py",
                "multimodal_mil_model.py", "training.py", "experiment_runner.py", "experiment_evaluation.py",
                "image_preprocessing.py", "mil_diagnostics.py"]
source_hashes = {name: file_sha256(ROOT / name) for name in source_files}
comparison_protocol = fingerprint({
    "reference_config": file_sha256(REFERENCE_TRIAL / "config.json"),
    "split": file_sha256(REFERENCE_DIR / "split_manifest.csv"),
    "sources": source_hashes, "procedure": "dual_fusion_softmax_and_paper_binary_macro_f1_v1",
    "baselines": file_sha256(BASELINE_CSV),
})
PROVENANCE = {
    "project_root": str(ROOT), "reference_dir": str(REFERENCE_DIR), "notebook_path": str(NOTEBOOK_PATH),
    "reference_scenario": "S26_pretrained_frozen_adam_P2", "comparison_protocol": comparison_protocol,
    "source_sha256": source_hashes,
    "reference_artifacts": {name: file_sha256(REFERENCE_DIR / name) for name in [
        "split_manifest.csv", "tabular_preprocessing.joblib", "image_stats.json", "cohort.csv"]},
    "initialization": "fresh model and optimizer; ImageNet frozen backbone; no trained parent weights",
}
print("Run:", RUN_DIR)
print(json.dumps(SETTINGS, indent=2))
# Checkpoint/early stopping: macro-F1 3 kelas untuk softmax, 2 kelas untuk binary.
# Pelaporan tetap mencatat AUROC, AP dan area PR trapezoid. Tidak mengganti monitor ke AUROC.


Run: d:\Kuliah\Magister\Semester 1\Topik Dalam Pengolahan Citra Digital\Multimodal MIL-Attention based Hierarchical Classification Regensburg Pediatric Dataset\experiments\Experiment 1\runs\severity_s26_20261018_131009_015915_seed1
{
  "seed": 1,
  "epochs": 100,
  "batch_size": 16,
  "num_workers": 0,
  "use_amp": true,
  "monitor": "val_f1",
  "patience": 25,
  "min_delta": 0.0,
  "threshold": 0.5,
  "early_stopping_rule": "best_monitor",
  "test_bootstrap_resamples": 1000,
  "device": "cuda",
  "save_plots": true,
  "verbose": true
}


In [3]:
# 3. Preprocessing dan pasien referensi, tanpa split atau fit ulang
datasets, feature_cols = load_reference_data(ROOT, REFERENCE_DIR)
assert {name: len(datasets[name]) for name in ("train", "val", "test")} == {"train": 369, "val": 93, "test": 116}
assert len(feature_cols) == 53 and "Length_of_Stay" in feature_cols
if VARIANT == "severity":
    # Paper Marcinkevics: complicated vs uncomplicated ATAU no appendicitis.
    # Karena itu tidak menyaring pasien no appendicitis dari train/val/test.
    print("Severity: semua pasien; positif=complicated, negatif=no/uncomplicated.")
elif VARIANT == "softmax":
    # Dua attention dan dua fusion tetap ada; dua output fusion digabung ke head 3 kelas.
    # Cross-entropy bersama; stream tidak lagi punya supervisi diagnosis/severity terpisah.
    print("Softmax: dual attention + dual fusion; prediksi tiga kelas memakai argmax.")
else:
    print("Diagnosis: semua pasien; positif=appendicitis.")
print({name: len(datasets[name]) for name in ("train", "val", "test")})


Severity: semua pasien; positif=complicated, negatif=no/uncomplicated.
{'train': 369, 'val': 93, 'test': 116}


In [4]:
# 4. Satu training dari awal, FP32 pada validation
result = train_comparison(
    RUN_DIR, VARIANT, MODEL_CONFIG, OPTIMIZER_CONFIG,
    datasets, SETTINGS, PROVENANCE,
)
print("Status:", result["status"], "| best epoch:", result["best_epoch"])
for split in ("train", "val"):
    m = result[split]["overall"] if VARIANT == "softmax" else result[split]
    print(split, {key: m.get(key) for key in ("accuracy", "macro_f1", "auroc", "auroc_macro_ovr", "auprc_ap", "auprc_trapezoid")})


{
  "variant": "severity",
  "model": {
    "num_tabular_features": 53,
    "use_dropout": true,
    "dropout_rate": 0.2,
    "backbone": "resnet18",
    "pretrained": true,
    "freeze_backbone": true,
    "backbone_trainable_stage": null,
    "image_embed_dim": null,
    "image_dropout": 0.2,
    "aggregation": "attention",
    "attention_fusion": "dual",
    "attention_dim": 128,
    "attention_gated": true,
    "attention_dropout": 0.2,
    "lstm_hidden": 256,
    "lstm_layers": 1,
    "lstm_bidirectional": false,
    "lstm_dropout": 0.2,
    "tabular_encoder": "mlp",
    "tabular_embed_dim": 64,
    "mlp_hidden_dims": [
      128
    ],
    "mlp_dropout": 0.2,
    "tabnet_n_steps": 5,
    "tabnet_gamma": 1.5,
    "tabnet_n_shared": 2,
    "tabnet_n_independent": 2,
    "tabnet_virtual_batch_size": 128,
    "tabnet_momentum": 0.02,
    "fusion_hidden_dims": [
      256,
      64
    ],
    "fusion_dropout": 0.2,
    "head": "dual_head",
    "complication_loss_weight": 1.0,
    "tab

In [6]:
# 5. Testing checkpoint validation dan tabel perbandingan
EVALUATE_TEST = True # True setelah selesai training; jalankan ulang cell ini saja.
if EVALUATE_TEST:
    test_report = test_comparison(RUN_DIR, datasets)
    print(pd.read_csv(RUN_DIR / "testing" / "overview.csv").to_string(index=False))
    comparisons = export_paper_comparisons(
        REFERENCE_DIR, RUNS_ROOT, RUN_DIR, BASELINE_CSV, comparison_protocol,
    )
    for name, frame in comparisons.items():
        print("\n", name)
        print(frame.to_string(index=False))
    print("Laporan perbandingan:", RUN_DIR / "comparisons" / "comparison.html")
else:
    print("Testing belum dijalankan. Checkpoint dipilih dari validation macro-F1.")
# Perbandingan mengambil run completed terbaru dengan protokol yang sama, bukan skor test terbaik.
# Model yang belum dilatih/diuji tetap pending; jalankan ulang cell ini setelah ketiganya selesai.
# Severity q pada true appendicitis dilaporkan terpisah dari target severity paper pada semua pasien.
# Softmax binary memakai marginal P(appendicitis)=P1+P2 dan P(complicated)=P2 pada threshold 0.5.
# Keputusan binary dari argmax softmax diekspor terpisah agar tidak tertukar dengan marginal.


          group status  patients  support_positive  support_negative  tp  tn  fp  fn  accuracy  precision      ppv      npv  sensitivity   recall  specificity       f1  negative_class_precision  negative_class_recall  negative_class_f1  macro_f1  weighted_f1      fpr      fnr  balanced_accuracy      mcc    auroc  auprc_average_precision    brier  log_loss  auprc_ap  auprc_trapezoid                         definition
            all     ok       116                19                97  16  91   6   3  0.922414   0.727273 0.727273 0.968085     0.842105 0.842105     0.938144 0.780488                  0.968085               0.938144           0.952880  0.866684     0.924643 0.061856 0.157895           0.890125 0.736582 0.940315                 0.836427 0.059763  0.214731  0.836427         0.832786 complicated_vs_no_or_uncomplicated
   single_image     ok        37                10                27   9  26   1   1  0.945946   0.900000 0.900000 0.962963     0.900000 0.900000     0.962963 0